# Video Stress Analysis Notebook

This notebook implements the video analysis logic to detect emotions and analyze stress levels from a video file.

In [ ]:
import cv2
import os
import sys
from deepface import DeepFace
from collections import Counter

# Setup paths to access the backend modules
current_dir = os.getcwd()
backend_path = os.path.abspath(os.path.join(current_dir, "..", "backend"))
if backend_path not in sys.path:
    sys.path.append(backend_path)

from app.core.config import vid_stress
print(f"Project path added: {backend_path}")
print(f"Test video path: {vid_stress}")

In [ ]:
def analyze_video_offline(video_path):
    """
    Analyzes a video to detect emotions and calculate stress/confidence scores.
    """
    # Check if file exists
    if not os.path.exists(video_path):
        print(f"Error: File {video_path} does not exist.")
        return None

    # Loading video
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30  # Default if undetermined
    
    frame_count = 0
    emotions_list = []

    print(f"--- Analyzing video: {video_path} ---")
    print("Processing (1 frame per second to optimize CPU)...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # CPU Optimization: Analyze only 1 frame per second
        if frame_count % fps == 0:
            try:
                # DeepFace Analysis
                results = DeepFace.analyze(
                    frame, 
                    actions=['emotion'], 
                    enforce_detection=False,
                    detector_backend='opencv'
                )
                
                # Get dominant emotion
                if isinstance(results, list):
                    dom_emotion = results[0]['dominant_emotion']
                else:
                    dom_emotion = results['dominant_emotion']
                
                emotions_list.append(dom_emotion)
                print(f"Second {frame_count//fps}: {dom_emotion}")
                
            except Exception as e:
                # Ignore frames where face is not detectable
                pass

        frame_count += 1
        
    cap.release()

    # Final calculations
    if not emotions_list:
        return {"error": "No face detected or unreadable video."}

    # Classification for scores
    stress_emotions = ["fear", "angry", "sad", "disgust"]
    confidence_emotions = ["happy", "neutral"]
    
    total_frames = len(emotions_list)
    stress_count = sum(1 for e in emotions_list if e in stress_emotions)
    confidence_count = sum(1 for e in emotions_list if e in confidence_emotions)
    
    emotion_dominant = Counter(emotions_list).most_common(1)[0][0]
    stress_score = round((stress_count / total_frames) * 100)
    confidence_score = round((confidence_count / total_frames) * 100)

    return {
        "emotion_dominant": emotion_dominant,
        "stress_score": f"{stress_score}%",
        "confidence_score": f"{confidence_score}%"
    }

In [ ]:
# Run Test
if vid_stress:
    result = analyze_video_offline(vid_stress)
    
    if result:
        print("\n" + "="*30)
        print("ANALYSIS RESULTS")
        print("="*30)
        print(f"Dominant Emotion : {result.get('emotion_dominant')}")
        print(f"Stress Score     : {result.get('stress_score')}")
        print(f"Confidence Score : {result.get('confidence_score')}")
        print("="*30)
else:
    print("No test video path found in configuration (vid_stress_env).")